# Player data quality control

Run this notebook before analyzing a player. Change `PLAYER` and, when appropriate, `EARLIEST_MATCH_DATE`. The notebook validates the Match Charting Project CSVs, displays every exclusion reason, and saves a reusable per-player snapshot.

Tier A defects drop a whole match (except orphan rows); parser and outcome defects drop only the affected point. Warnings do not remove data.

In [ ]:
# Player-specific configuration
PLAYER = "Carlos Alcaraz"
# Set this to the player's first plausible chart date. Use None to disable only
# the lower-bound check; malformed and implausibly future dates are still rejected.
EARLIEST_MATCH_DATE = "2020-01-01"
COVERAGE_WARNING_THRESHOLD = 0.80

DRIVE_ROOT = "/content/drive/MyDrive/projSwing"
MATCHES_PATH = f"{DRIVE_ROOT}/charting-m-matches.csv"
POINT_PATHS = [
    f"{DRIVE_ROOT}/charting-m-points-2010s.csv",
    f"{DRIVE_ROOT}/charting-m-points-2020s.csv",
]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
import urllib.request
from pathlib import Path

repo_dir = Path(DRIVE_ROOT)
module_dir = repo_dir if (repo_dir / "data_qc.py").exists() else Path("/content")
module_path = module_dir / "data_qc.py"
if not module_path.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/mchadw/projSwing/main/data_qc.py",
        module_path,
    )
sys.path.insert(0, str(module_dir))

from data_qc import player_slug, run_qc_from_csv, save_artifacts

In [ ]:
result = run_qc_from_csv(
    MATCHES_PATH,
    POINT_PATHS,
    PLAYER,
    earliest_match_date=EARLIEST_MATCH_DATE,
    coverage_warning_threshold=COVERAGE_WARNING_THRESHOLD,
)

print(f"{PLAYER}: {len(result.match_meta):,} clean matches, "
      f"{len(result.player_points):,} clean points")
display(result.exclusion_summary)

print("Exclusion details")
display(result.exclusions)

print("Non-dropping coverage warnings")
display(result.warnings)

In [ ]:
QC_ARTIFACT_DIR = repo_dir / "qc" / player_slug(PLAYER)
manifest_path = save_artifacts(
    result,
    QC_ARTIFACT_DIR,
    source_paths=[MATCHES_PATH, *POINT_PATHS],
)
print(f"Saved QC snapshot: {manifest_path}")

## Reuse later

The output folder contains `match_meta.csv`, `player_points.csv`, `qc_exclusions.csv`, `qc_warnings.csv`, and `qc_manifest.json`. Future analysis can load the exact snapshot with:

```python
from data_qc import load_artifacts
clean = load_artifacts(QC_ARTIFACT_DIR)
match_meta = clean.match_meta
player_points = clean.player_points
```

The existing Alcaraz analysis notebook is intentionally not wired to these artifacts yet.